In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [ ]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [ ]:
wiki_tool.name

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings


loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
faiss_db = FAISS.from_documents(documents, OpenAIEmbeddings())
retriever = faiss_db.as_retriever()
retriever

Creating retrival tools

In [ ]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever=retriever, 
                                               name="wiki_retriever", 
                                               description="Useful for when you need to answer questions about Artificial Intelligence.")

In [ ]:
retriever_tool.name

Arxiv wrapper

In [ ]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv_tool.name

Combine all tools

In [ ]:
tools = [wiki_tool, retriever_tool, arxiv_tool]

In [ ]:
tools

Agents

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()


In [ ]:
from langchain_community.chat_models import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [ ]:
#creating prompts
from langsmith import Client

client = Client()
prompt = client.pull_prompt("pathfinderai/openai-functions-agent")
prompt.messages

In [ ]:
from langchain_classic.agents import (
    AgentExecutor,
    create_openai_tools_agent,
)
agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompt)


In [ ]:
#Agent Excetur
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

In [ ]:
agent_executor.invoke({"input": "What is Artificial Intelligence?", "chat_history": []})

In [ ]:
agent_executor.invoke({"input": "What is the paper 1605.08386 about?", "chat_history": []})

In [ ]:
agent_executor.invoke({"input": "What is Artificial Intelligence?", "chat_history": []})